In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_from_disk
from transformers import MT5ForConditionalGeneration, MT5Tokenizer
import evaluate
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

In [ ]:
MODEL_PATH = "../models/mt5-haoussa-zarma"
TEST_PATH = "../data/processed/test_dataset"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
model = MT5ForConditionalGeneration.from_pretrained(MODEL_PATH)
tokenizer = MT5Tokenizer.from_pretrained(MODEL_PATH)

model.to(device)
model.eval()

In [ ]:
test_dataset = load_from_disk(TEST_PATH)

print("Test size:", len(test_dataset))
test_dataset[0]

In [ ]:
def translate(text):
    input_text = "translate Hausa to Zarma: " + text

    inputs = tokenizer(input_text, return_tensors="pt", truncation=True).to(device)

    outputs = model.generate(
        **inputs,
        max_length=32,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
predictions = []
references = []
sources = []

for sample in test_dataset:
    input_ids = sample["input_ids"]
    label_ids = sample["labels"]

    source = tokenizer.decode(input_ids, skip_special_tokens=True)
    target = tokenizer.decode(label_ids, skip_special_tokens=True)

    pred = translate(source.replace("translate Hausa to Zarma: ", ""))

    predictions.append(pred)
    references.append(target)
    sources.append(source)

In [ ]:
bleu = evaluate.load("sacrebleu")

bleu_score = bleu.compute(
    predictions=predictions,
    references=[[ref] for ref in references]
)

print("BLEU:", bleu_score["score"])

In [ ]:
rouge = evaluate.load("rouge")

rouge_score = rouge.compute(
    predictions=predictions,
    references=references
)

print(rouge_score)

In [ ]:
chrf = evaluate.load("chrf")

chrf_score = chrf.compute(
    predictions=predictions,
    references=references
)

print("chrF:", chrf_score["score"])

In [ ]:
bleu_sentence_scores = []

for pred, ref in zip(predictions, references):
    score = bleu.compute(
        predictions=[pred],
        references=[[ref]]
    )["score"]
    
    bleu_sentence_scores.append(score)

plt.figure()
sns.histplot(bleu_sentence_scores, bins=20)
plt.title("Distribution BLEU par phrase")
plt.show()

In [ ]:
results_df = pd.DataFrame({
    "source": sources,
    "reference": references,
    "prediction": predictions
})

results_df.head(20)

In [ ]:
errors = results_df[results_df["reference"] != results_df["prediction"]]

errors.sample(10)

In [ ]:
results_df["pred_len"] = results_df["prediction"].apply(lambda x: len(x.split()))
results_df["ref_len"] = results_df["reference"].apply(lambda x: len(x.split()))

plt.figure()
sns.scatterplot(x="ref_len", y="pred_len", data=results_df)
plt.title("Longueur référence vs prédiction")
plt.show()

In [ ]:
"""
Types d'erreurs observées:

1. Traduction littérale incorrecte
2. Mot manquant
3. Mauvais ordre des mots
4. Confusion de synonymes
5. Répétition ou hallucination
"""

In [ ]:
test_sentences = [
    "Sannu",
    "Ina gidan ku?",
    "Nagode",
    "Ina jin yunwa",
    "Ina zuwa kasuwa"
]

for sentence in test_sentences:
    print("Hausa:", sentence)
    print("Zarma:", translate(sentence))
    print("-" * 40)

In [ ]:
print(f"BLEU Score : {bleu_score['score']:.2f}")
print(f"chrF Score : {chrf_score['score']:.2f}")

In [ ]:
"""
Observations:

1. BLEU score modéré → dataset limité
2. chrF plus fiable pour langues locales
3. Modèle performant sur phrases simples
4. Difficulté sur phrases longues ou complexes
5. Bonne mémorisation des expressions fréquentes

Conclusion:

Le modèle démontre une capacité de traduction basique Hausa → Zarma.
Cependant, les performances sont limitées par la taille du dataset.

Améliorations possibles:
- Augmenter les données
- Ajouter phrases complexes
- Utiliser back-translation
"""